# 19 — Prompting for Coding Agents

## Scenario
Autonomous coding agents (like Devin, Copilot Workspace, or advanced IDE extensions) are extremely powerful. They can read your entire codebase and execute terminal commands.

**The Problem:** If you give an autonomous agent a vague prompt like `"Fix the authentication bug"`, it might decide the best way to fix it is to rewrite your entire database layer, breaking 50 other things in the process.

**The Solution:** We must shift from "writing prompts" to "writing Engineering Contracts."

In [ ]:
import os
from typing import List
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'


## Step 1: The Vague Prompt (Danger)

Let's simulate what a coding agent might do if given a vague prompt.

In [ ]:
vague_prompt = "Fix the authentication bug where users can't log in."

# Simulating the Agent's plan
agent_system_instruction = """\nYou are an autonomous coding agent with full repository access.\nBased on the user's prompt, outline your execution plan. Be ambitious.\n"""

response_vague = client.models.generate_content(
    model=MODEL_ID,
    contents=vague_prompt,
    config=types.GenerateContentConfig(
        system_instruction=agent_system_instruction,
        temperature=0.7,
    )
)

print("--- AGENT PLAN (VAGUE PROMPT) ---")
print(response_vague.text)

# Notice how the agent likely proposes checking the database, rewriting the auth middleware,
# and updating the frontend. This is "scope creep" and it is dangerous.

## Step 2: The Engineering Contract

To control the agent, we (the humans) should fill out a strict Pydantic contract before handing the task over. This explicitly limits the agent's scope.

In [ ]:
class CodingAgentContract(BaseModel):
    problem_statement: str = Field(description="Exact description of the bug or feature.")
    file_scope: List[str] = Field(description="The ONLY files the agent is allowed to read or modify.")
    tests_to_run: str = Field(description="The exact terminal command to verify success.")
    completion_criteria: str = Field(description="What must be true for the task to be considered done.")

# The Human fills this out
my_contract = CodingAgentContract(
    problem_statement="Users cannot log in because the JWT token expiration is set to 0 seconds instead of 3600.",
    file_scope=["src/auth/jwt_utils.py"],
    tests_to_run="pytest tests/auth/test_jwt.py",
    completion_criteria="The test suite passes and the token expiration is 3600."
)


## Step 3: Prompting with the Contract

Now we pass the structured JSON contract to the agent, instructing it to strictly obey the boundaries.

In [ ]:
contract_prompt = f"""\nExecute the following task contract.\nCRITICAL RULE: You may NOT modify or read any files outside of the `file_scope`.\n\nCONTRACT:\n{my_contract.model_dump_json(indent=2)}\n"""

response_contract = client.models.generate_content(
    model=MODEL_ID,
    contents=contract_prompt,
    config=types.GenerateContentConfig(
        system_instruction=agent_system_instruction,
        temperature=0.0,
    )
)

print("\n--- AGENT PLAN (STRUCTURED CONTRACT) ---")
print(response_contract.text)

# Notice how the agent's plan is now highly constrained, focused solely on the single file
# and the specific pytest command. This is how you safely manage autonomous coding agents.